# 03 - Modeling

**Purpose:** Establish a baseline Random Forest model for bulk modulus prediction using GroupKFold cross-validation grouped by chemical system, then run an ablation study comparing feature sets (MAGPIE only vs. + crystal system vs. + VEC) and model types (Random Forest vs. Lasso).

**Input:** `../data/bradley_intermetallics_featured.csv`

## Baseline Random Forest (GroupKFold by chemical system)

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold

featured_df = pd.read_csv("../data/bradley_intermetallics_featured.csv")

target_column = "bulk_modulus_vrh"
group_column = "chem_system"

# Columns that are identifiers/metadata rather than model features
non_feature_columns = {"material_id", "formula", "elements", "crystal_system", "num_elements", target_column, group_column}
crystal_system_columns = [c for c in featured_df.columns if c.startswith("cs_")]
magpie_columns = [c for c in featured_df.columns if c not in non_feature_columns and c not in crystal_system_columns and c != "VEC"]

# Feature sets used again in the ablation study below
feature_sets = {
    "magpie_only": magpie_columns,
    "magpie_plus_crystal": magpie_columns + crystal_system_columns,
    "magpie_plus_crystal_plus_vec": magpie_columns + crystal_system_columns + ["VEC"],
}

baseline_features = feature_sets["magpie_plus_crystal_plus_vec"]
X_baseline = featured_df[baseline_features].fillna(0)
y = featured_df[target_column]
chem_system_groups = featured_df[group_column]

# GroupKFold grouped by chem_system prevents compounds from the same chemical system (which share
# nearly identical MAGPIE features) from appearing in both the train and test folds
group_kfold = GroupKFold(n_splits=5)
baseline_rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)

fold_r2_scores = []
for fold_number, (train_idx, test_idx) in enumerate(group_kfold.split(X_baseline, y, chem_system_groups), start=1):
    baseline_rf.fit(X_baseline.iloc[train_idx], y.iloc[train_idx])
    fold_r2 = baseline_rf.score(X_baseline.iloc[test_idx], y.iloc[test_idx])
    fold_r2_scores.append(fold_r2)
    print(f"Fold {fold_number}: R^2 = {fold_r2:.3f}")

fold_r2_scores = np.array(fold_r2_scores)
print("\nBaseline Random Forest (MAGPIE + crystal system + VEC features)")
print(f"GroupKFold (5 splits, grouped by chem_system) mean R^2 = {fold_r2_scores.mean():.3f} +/- {fold_r2_scores.std():.3f}")

Fold 1: R^2 = 0.805
Fold 2: R^2 = 0.696
Fold 3: R^2 = 0.748
Fold 4: R^2 = 0.792
Fold 5: R^2 = 0.813

Baseline Random Forest (MAGPIE + crystal system + VEC features)
GroupKFold (5 splits, grouped by chem_system) mean R^2 = 0.771 +/- 0.043


## Ablation Study: Feature Sets vs Model Type

In [2]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ablation_results = []
for feature_set_name, feature_columns in feature_sets.items():
    X_set = featured_df[feature_columns].fillna(0)

    rf_fold_scores = []
    lasso_fold_scores = []
    for train_idx, test_idx in group_kfold.split(X_set, y, chem_system_groups):
        X_train, X_test = X_set.iloc[train_idx], X_set.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        rf_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        rf_model.fit(X_train, y_train)
        rf_fold_scores.append(rf_model.score(X_test, y_test))

        # Lasso needs standardized inputs since it penalizes coefficient magnitude directly
        lasso_model = make_pipeline(StandardScaler(), Lasso(alpha=0.5, max_iter=10000))
        lasso_model.fit(X_train, y_train)
        lasso_fold_scores.append(lasso_model.score(X_test, y_test))

    ablation_results.append({
        "feature_set": feature_set_name,
        "n_features": len(feature_columns),
        "RF_R2_mean": np.mean(rf_fold_scores),
        "RF_R2_std": np.std(rf_fold_scores),
        "Lasso_R2_mean": np.mean(lasso_fold_scores),
        "Lasso_R2_std": np.std(lasso_fold_scores),
    })

ablation_summary_df = pd.DataFrame(ablation_results)
print("Ablation Study: GroupKFold (5 splits, grouped by chem_system)\n")
print(ablation_summary_df.to_string(index=False))

Ablation Study: GroupKFold (5 splits, grouped by chem_system)

                 feature_set  n_features  RF_R2_mean  RF_R2_std  Lasso_R2_mean  Lasso_R2_std
                 magpie_only         132    0.757016   0.040566       0.722989      0.044430
         magpie_plus_crystal         139    0.770849   0.043744       0.725608      0.044120
magpie_plus_crystal_plus_vec         140    0.770815   0.043412       0.725812      0.043958
